In [ ]:
import logging
from logging.handlers import RotatingFileHandler
import psycopg2
import pandas as pd
import dash
from dash import dcc, html
import dash_core_components as dcc
import dash_bootstrap_components as dbc
from dash.dependencies import Input, Output
import plotly.graph_objects as go
from configparser import ConfigParser

# Configure logging with rotating file handler
log_file = "api/logs/test.log"
handler = RotatingFileHandler(log_file, maxBytes=5000000, backupCount=5)  # 5 MB max per log file, keep 5 backups
formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
handler.setFormatter(formatter)

# Root logger
logger = logging.getLogger()
logger.setLevel(logging.INFO)
logger.addHandler(handler)

logger.info("Starting database connection script.")

# Fetch database config
def configDB(filename="api/ini/NSE.ini", section="postgresql"):
    parser = ConfigParser()
    parser.read(filename)
    db = {}
    if parser.has_section(section):
        params = parser.items(section)
        for param in params:
            db[param[0]] = param[1]
    else:
        logger.error(f"Section {section} not found in {filename} file.")
        raise Exception(f'Section {section} not found in {filename} file.')
    return db

# Test configDB function
print("Testing configDB function...")
db_config = configDB()
print("Database Config:", db_config)
print("configDB function test complete.\n")

In [106]:
# Connect to PostgreSQL database and fetch data
def connect(table_name):
    conn = None
    try:
        # Convert table_name to uppercase for case-insensitive queries
        table_name = table_name.upper()

        # Load connection parameters
        params = configDB()
        
        # Connect to PostgreSQL
        logger.info('Connecting to the PostgreSQL database...')
        conn = psycopg2.connect(**params)

        # Create a cursor object
        cur = conn.cursor()

        # Query to fetch data from the specified table
        cur.execute(f'SELECT * FROM "{table_name}";')  # Fetch the top 100 rows
        # cur.execute(f'SELECT * FROM "{table_name}" LIMIT 100;')  # Fetch the top 100 rows

        # Fetch column names
        colnames = [desc[0] for desc in cur.description]
        
        # Fetch data from the query
        rows = cur.fetchall()

        # Convert the fetched data into a pandas DataFrame
        df = pd.DataFrame(rows, columns=colnames)
        
        # Close the communication with the PostgreSQL database
        cur.close()
        return df

    except (Exception, psycopg2.DatabaseError) as error:
        logger.error(f"Error: {error}")
        print(f"Error: {error}")
        return None
    finally:
        if conn is not None:
            conn.close()
            logger.info('Database connection closed.')

# Dash App
app = dash.Dash(__name__)

In [108]:
# Fetch all table names from PostgreSQL
def get_table_names():
    conn = None
    try:
        # Load connection parameters
        params = configDB()
        
        # Connect to PostgreSQL
        conn = psycopg2.connect(**params)
        cur = conn.cursor()

        # Query to list all tables in the current database
        cur.execute("""
            SELECT table_name 
            FROM information_schema.tables 
            WHERE table_schema = 'public';
        """)
        
        # Fetch the list of tables
        tables = cur.fetchall()
        table_list = [table[0] for table in tables]
        return table_list

    except (Exception, psycopg2.DatabaseError) as error:
        logger.error(f"Error: {error}")
        print(f"Error: {error}")
        return []
    finally:
        if conn is not None:
            conn.close()

# Dash App
app = dash.Dash(__name__)

# Layout of the Dash App
app.layout = html.Div([
    html.H1("Candlestick Chart from PostgreSQL Database"),

    # Dropdown to select table
    dcc.Dropdown(
        id='table-dropdown',
        options=[{'label': table, 'value': table} for table in get_table_names()],
        placeholder="Select a table",
    ),

    # Graph to display the candlestick chart
    dcc.Graph(id='candlestick-chart')
])

# Callback to update the chart based on selected table
@app.callback(
    Output('candlestick-chart', 'figure'),
    [Input('table-dropdown', 'value')]
)
def update_chart(table_name):
    if table_name:
        # Fetch the data from the selected table
        df = connect(table_name)

        # Ensure the DataFrame has necessary columns for candlestick chart
        if df is not None and all(col in df.columns for col in ['timestamp', 'open_price', 'high_price', 'low_price', 'close_price']):
            # Create the candlestick chart using Plotly
            fig = go.Figure(data=[go.Candlestick(x=df['timestamp'],
                                                 open=df['open_price'],
                                                 high=df['high_price'],
                                                 low=df['low_price'],
                                                 close=df['close_price'])])
            # Update layout with title
            fig.update_layout(title=f'Candlestick Chart for {table_name}',
                              xaxis_title='Timestamp',
                              yaxis_title='Price',
                              xaxis_rangeslider_visible=False)
            return fig
        else:
            return go.Figure()  # Return empty figure if columns are missing
    return go.Figure()  # Return empty figure initially

# Run the app
if __name__ == '__main__':
    app.run_server(debug=True)